# Notebook 03: Dynamic Few-Shot Retrieval and Diversity-Aware Selection
### Embedding Generation, FAISS Indexing, and Retrieval-Augmented In-Context Learning

This notebook investigates retrieval-augmented dynamic prompting:
1. Dense vector representation via `sentence-transformers/all-MiniLM-L6-v2`.
2. FAISS `IndexFlatIP` indexing of the 9,002 development pool samples.
3. K-shot ablation across $K \in \{1, 3, 5, 8, 10\}$.
4. Diversity-Aware Retrieval vs Simple Similarity Retrieval.
5. Accuracy, latency, token consumption, and cost tradeoffs.


In [ ]:
# ==========================================
# 0. Google Colab / Local Environment Setup
# ==========================================
import sys, os
from pathlib import Path

# If running in Google Colab, install repository and dependencies
if "google.colab" in sys.modules:
    print("Detected Google Colab environment. Setting up...")
    !git clone https://github.com/your-username/banking-llm-optimizer.git
    %cd banking-llm-optimizer
    !pip install -r requirements.txt
    
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
else:
    print("Running in local environment.")
    ROOT_DIR = Path(".").resolve()
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))


### 1. Load Precomputed FAISS Index and Initialize Pipeline


In [ ]:
from src.data.loader import BankingDataLoader
from src.pipeline import BankingIntentPipeline
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.cost_analysis import CostAnalyzer
import pandas as pd
import matplotlib.pyplot as plt

loader = BankingDataLoader()
train_pool, val_df, test_df = loader.load_processed_splits()
eval_val = val_df.head(500).copy()

pipeline = BankingIntentPipeline()
print(f"FAISS index loaded with {pipeline.index_manager.index.ntotal} vectors.")


### 2. K-Shot Ablation Experiment (K = 1, 3, 5, 8, 10)


In [ ]:
k_values = [1, 3, 5, 8, 10]
k_results = []
cost_analyzer = CostAnalyzer()

for k in k_values:
    print(f"Evaluating Dynamic Few-Shot with K={k}...")
    df_k = pipeline.evaluate_dataset(eval_val, strategy="dynamic_few_shot", k=k)
    m = ClassificationMetrics.compute_all_metrics(df_k["true_intent"].tolist(), df_k["predicted_intent"].tolist())
    c = cost_analyzer.summarize_benchmark_run(df_k["latency_ms"].tolist(), df_k["input_tokens"].tolist(), df_k["output_tokens"].tolist())
    k_results.append({
        "K": k,
        "Accuracy": round(m["accuracy"], 4),
        "Macro-F1": round(m["macro_f1"], 4),
        "Weighted-F1": round(m["weighted_f1"], 4),
        "Avg Tokens": round(c["avg_total_tokens"], 1),
        "P95 Latency": round(c["latency_p95_ms"], 1),
        "Cost": round(c["cost_per_1k_queries_usd"], 4)
    })

k_df = pd.DataFrame(k_results)
k_df.to_csv("results/tables/dynamic_few_shot_results.csv", index=False)
display(k_df)


### 3. Diversity-Aware Retrieval vs Simple Similarity Retrieval


In [ ]:
print("Running Diversity-Aware Dynamic Few-Shot (K=5)...")
div_df = pipeline.evaluate_dataset(eval_val, strategy="dynamic_few_shot", k=5, diversity=True)
div_metrics = ClassificationMetrics.compute_all_metrics(div_df["true_intent"].tolist(), div_df["predicted_intent"].tolist())

sim_k5 = k_df[k_df["K"] == 5].iloc[0]
print(f"Similarity-Only K=5 Macro-F1: {sim_k5['Macro-F1']}")
print(f"Diversity-Aware K=5 Macro-F1: {div_metrics['macro_f1']:.4f}")


### 4. Plot K vs Macro-F1 and Token Consumption Curves


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

color = "tab:blue"
ax1.set_xlabel("Number of Retrieved Exemplars (K)")
ax1.set_ylabel("Macro-F1 Score", color=color)
ax1.plot(k_df["K"], k_df["Macro-F1"], marker="o", color=color, linewidth=2, label="Macro-F1")
ax1.tick_params(axis="y", labelcolor=color)
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
color = "tab:red"
ax2.set_ylabel("Average Total Tokens", color=color)
ax2.plot(k_df["K"], k_df["Avg Tokens"], marker="s", color=color, linestyle="--", linewidth=2, label="Tokens")
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Effect of Demonstration Scale (K) on Macro-F1 and Token Overhead")
plt.tight_layout()
plt.savefig("results/figures/k_vs_performance.png", dpi=300)
plt.show()
